# WorldQuant BRAIN Alpha 遍历 V2.1

目标：**优先提高可提交 Alpha 的产出效率**。默认研究模式为 `POWER_POOL_ATOM`，同时保留 `ATOM / POWER_POOL / REGULAR`。

主流程：Data Fields → Data Coverage Filter → Stage1 Core → Targeted Extension → Stage2 Neutralize → Group-Rank Fallback → Live Check → Fail-aware Repair → Final Check。

V2.1 原则：
- 不改变 `simulation_key = expression + settings`，继续复用原 `alpha_results.db`。
- 不写死 Power Pool 支持地区；平台实时 eligibility/check 为最终依据。
- Data Coverage 阈值由 Notebook 配置，默认 `>= 0.90`。
- GLB / 其他地区的并发上限也由 Notebook 配置，默认分别 4 / 8。
- 不加入 Dataset Historical Coverage Preflight。


## 1. 环境初始化

加载 V2.1 库。请把 `machine_lib_V2_1.py`、本 Notebook 和原 `alpha_results.db` 放在同一项目目录。


In [1]:
# 加载依赖和 V2.1 模块，并确认当前 Python 与项目目录。
import sys
import importlib
from pathlib import Path

import pandas as pd
import machine_lib_V2_1 as machine_lib

# 重新加载本地库，确保刚修改的 .py 立即生效。
machine_lib = importlib.reload(machine_lib)
from machine_lib_V2_1 import *

# 记录项目路径，并统一设置结果表的显示宽度。
module_path = Path(machine_lib.__file__).resolve()
project_dir = module_path.parent
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 50)

print("V2.1 library:", module_path)
print("Project:", project_dir)
print("Python:", sys.executable)


V2.1 library: F:\一二三\v2\machine_lib_V2_1.py
Project: F:\一二三\v2
Python: c:\Users\Administrator\AppData\Local\Programs\Python\Python39\python.exe


## 2. 参数配置

只需要优先检查这一格。所有阈值和并发限制都在这里配置，不隐藏在 Notebook 逻辑里。


In [16]:
# 本格集中配置研究目标、BRAIN 参数、字段筛选、并发和各阶段开关。
# =========================
# 研究目标
# =========================
# 选择本轮主要目标模式；默认优先寻找 PP + ATOM。
TARGET_MODE = "POWER_POOL_ATOM"   # POWER_POOL_ATOM / ATOM / POWER_POOL / REGULAR

# =========================
# BRAIN 回测设置
# =========================
# 设置本轮 BRAIN 回测环境；Power Pool 支持地区不在本地写死。
REGION = "GLB"                    # 不在本地写死 Power Pool 支持地区
UNIVERSE = "TOPDIV3000"
DELAY = 1
DATASET_ID = "risk60"
NEUTRALIZATION = "SUBINDUSTRY"
INIT_DECAY = 4
TRUNCATION = 0.08
TEST_PERIOD = "P0Y"

# =========================
# 数据字段筛选
# =========================
# 只让达到 Coverage 阈值的字段进入正式搜索，阈值可自行调整。
MIN_DATA_COVERAGE = 0.90           # 可改；>= 该值才进入 Stage1
DATA_COVERAGE_COLUMN = None        # None=自动识别 API 中真正有效的 coverage 列
MAX_FIELDS = None                  # 在 Coverage + 人工排除之后再截取

# 人工排除只是额外保护，不等同于 Dataset 历史覆盖预检。
MANUAL_EXCLUDED_FIELDS_BY_DATASET = {
    "other169": {
        "bid_price_percent_of_notional",
        "interest_rate_curve_parallel_shift_impact",
        "isda_assumed_recovery_percent",
        "mid_par_spread_basis_points",
        "mid_quote_spread_basis_points",
        "oth169_monthlyobservationscore",
        "oth169_percentofparoffer",
        "oth169_recrisk",
    },
}

# =========================
# 并发与轮询设置
# =========================
# 设置期望并发数；底层会按地区上限自动收紧。
CONCURRENCY = 4                    # 你希望开的并发数
GLB_MAX_CONCURRENCY = 4            # 当前 GLB 上限，可改
OTHER_MAX_CONCURRENCY = 8          # 当前非 GLB 上限，可改
POLL_TIMEOUT_SECONDS = 1800        # 30分钟；超时保留 simulation_url，后续 Resume

# =========================
# 各阶段筛选阈值
# =========================
STAGE1_MIN_ABS_SHARPE = 0.80
STAGE1_MIN_ABS_FITNESS = 0.45
STAGE1_EXPLORATION_SHARPE = 0.55
STAGE1_EXPLORATION_FITNESS = 0.25
STAGE1_MIN_POSITIONS = 100

# Stage1 对 PP 只做“潜力门”，允许约 0.8 的信号进入 Stage2；
# Stage2 / Final 再把 PP 本地 Sharpe 预筛提高到 1.0。
STAGE1_PP_SHARPE_GATE = 0.80
FINAL_PP_SHARPE_GATE = 1.00

STAGE2_MIN_ABS_SHARPE = 1.00
STAGE2_MIN_ABS_FITNESS = 0.60
STAGE2_MIN_POSITIONS = 100
STAGE2_FALLBACK_MIN_SELECTED = 3

# =========================
# 各阶段执行开关
# =========================
# 第一次检查 Notebook 时建议先保持 False，确认字段筛选无误后再逐阶段开启。
RUN_STAGE1_CORE = True
RUN_STAGE1_EXTENDED = True
RUN_STAGE2_NEUTRALIZE = True
RUN_STAGE2_GROUP_RANK_FALLBACK = False
RUN_PRE_REPAIR_CHECK = False       # Regional Repair V2 不依赖此开关
RUN_REPAIR = True
RUN_FINAL_CHECK = True             # 只 GET /check，不提交 Alpha

CACHE_DB = str(project_dir / "alpha_results.db")

# 规范化模式并计算当前地区的实际并发数。
TARGET_MODE = normalize_target_mode(TARGET_MODE)
EFFECTIVE_CONCURRENCY = resolve_concurrency(
    REGION,
    CONCURRENCY,
    glb_max=GLB_MAX_CONCURRENCY,
    other_max=OTHER_MAX_CONCURRENCY,
    announce=False,
)

print("Target:", TARGET_MODE)
print("BRAIN:", REGION, UNIVERSE, "D" + str(DELAY), DATASET_ID, NEUTRALIZATION)
print("Data Coverage >=", MIN_DATA_COVERAGE)
print("Concurrency requested/effective:", CONCURRENCY, "/", EFFECTIVE_CONCURRENCY)
print("CACHE_DB:", CACHE_DB)


Target: POWER_POOL_ATOM
BRAIN: GLB TOPDIV3000 D1 risk60 SUBINDUSTRY
Data Coverage >= 0.9
Concurrency requested/effective: 4 / 4
CACHE_DB: F:\一二三\v2\alpha_results.db


## 3. 登录与数据字段筛选

先读取全部字段，再做 **Data Coverage → 人工排除 → MAX_FIELDS**。`MAX_FIELDS` 不会被低 Coverage 字段占掉。


In [37]:
# 登录 BRAIN，后续数据读取和回测请求都复用这个会话。
s = login()
print("BRAIN login: OK")


Loading BRAIN credentials from: F:\一二三\v2\key.txt
Logged in successfully.
BRAIN login: OK


In [ ]:
# 读取当前 Dataset 的全部字段，再依次执行 Coverage、人工排除和数量限制。
# 从 BRAIN 拉取当前 Dataset 的原始 Data Fields。
df_raw = get_datafields(
    s,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    delay=DELAY,
)

# 第一步：先按 Data Coverage 筛选
_df_cov, coverage_report = filter_datafields(
    df_raw,
    min_data_coverage=MIN_DATA_COVERAGE,
    coverage_column=DATA_COVERAGE_COLUMN,
)

# 第二步：再应用人工排除名单
manual_excluded = MANUAL_EXCLUDED_FIELDS_BY_DATASET.get(DATASET_ID, set())
if manual_excluded:
    df_fields = _df_cov[~_df_cov["id"].astype(str).isin(manual_excluded)].copy()
else:
    df_fields = _df_cov.copy()

# 第三步：最后再限制字段数量，避免低 Coverage 字段占名额
if MAX_FIELDS is not None:
    df_fields = df_fields.head(max(0, int(MAX_FIELDS))).copy()

# 整理最终字段表，并为 MATRIX / VECTOR 生成可用于表达式的字段记录。
df_fields = df_fields.reset_index(drop=True)
field_records = prepare_fields(df_fields)

# 输出筛选统计和字段预览，第一次运行时重点检查 Coverage 是否识别正确。
print("Raw Data Fields:", len(df_raw))
print("Coverage column chosen:", coverage_report["coverage_column"])
print("Coverage filtered out:", coverage_report["filtered_out"])
print("Manual excluded after coverage:", len(_df_cov) - len(df_fields) if MAX_FIELDS is None else "see table")
print("Final Data Fields entering Stage1:", len(df_fields))
print("Prepared expressions (VECTOR may x2):", len(field_records))

show_cols = [c for c in (
    "id", "type", "_data_coverage", "dateCoverage", "coverage",
    "pyramidMultiplier", "themes", "dateCreated"
) if c in df_fields.columns]
display(df_fields[show_cols].head(50))

display(pd.DataFrame(field_records)[[
    c for c in (
        "dataset_id", "field", "field_type", "vector_op", "data_coverage",
        "pyramid_multiplier", "expr"
    ) if c in pd.DataFrame(field_records).columns
]].head(20))


## 4. 第一阶段 — 核心搜索

核心算子保持：`raw / rank / zscore / ts_mean / ts_rank / ts_zscore / ts_delta / ts_std_dev`。核心窗口只跑 `5 / 22 / 66`；`120` 不在这一轮。


In [ ]:
# 生成 Stage1 Core 候选，并在回测前完成目标类型标记和上下文校验。
# 使用核心算子与核心窗口生成第一轮候选。
stage1_core_candidates = first_order_candidates(
    field_records,
    ts_operators=CORE_TS_OPS,
    cross_ops=("rank", "zscore"),
    init_decay=INIT_DECAY,
)
stage1_core_candidates = annotate_candidates(stage1_core_candidates, TARGET_MODE)
validate_candidate_context(
    stage1_core_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

print("Stage1 Core candidates:", len(stage1_core_candidates))
print("Has window=120:", any(c.get("window") == 120 for c in stage1_core_candidates))
preview = pd.DataFrame(stage1_core_candidates)
display(preview[[c for c in (
    "dataset_id", "field", "operator", "window", "vector_op",
    "operator_count", "data_field_count", "pp_structure_ok",
    "atom_structure_ok", "expr"
) if c in preview.columns]].head(20))

# 检查 SQLite 中已有缓存，确认哪些候选可直接复用、哪些仍需回测。
stage1_core_resume = resume_summary(
    stage1_core_candidates,
    neutralization=NEUTRALIZATION,
    region=REGION,
    universe=UNIVERSE,
    cache_db=CACHE_DB,
    delay=DELAY,
    truncation=TRUNCATION,
    test_period=TEST_PERIOD,
)


In [ ]:
# 只有打开 Stage1 Core 开关时才真正回测；否则仅生成候选供检查。
if RUN_STAGE1_CORE:
    stage1_core_results = simulate_candidates(
        stage1_core_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage1_core_results = pd.DataFrame()
    print("Stage1 Core 未启动：RUN_STAGE1_CORE=False")


In [ ]:
# 对 Stage1 Core 结果分层，并挑出值得进入定向扩展的字段。
stage1_core_classified = pd.DataFrame()
stage1_extension_fields = []

# 没有结果时不继续扩展；有结果时再进行信号分类。
if stage1_core_results.empty:
    print("暂无 Stage1 Core results。")
else:
    stage1_core_classified = classify_stage1_results(
        stage1_core_results,
        strict_abs_sharpe=STAGE1_MIN_ABS_SHARPE,
        strict_abs_fitness=STAGE1_MIN_ABS_FITNESS,
        min_positions=STAGE1_MIN_POSITIONS,
        exploration_abs_sharpe=STAGE1_EXPLORATION_SHARPE,
        exploration_abs_fitness=STAGE1_EXPLORATION_FITNESS,
    )
    summary = stage1_classification_summary(stage1_core_classified)
    print("Stage1 Core funnel:", summary)

    # 从当前 Candidate 元数据读取字段，避免旧缓存中的元数据污染。
    stage1_core_classified["field"] = stage1_core_classified["candidate"].map(
        lambda c: c.get("field") if isinstance(c, dict) else None
    )
    # 将严格通过、正向、可翻转负向和探索类信号都纳入扩展候选。
    active_mask = (
        stage1_core_classified["is_strict_pass"]
        | stage1_core_classified["is_positive"]
        | stage1_core_classified["is_flippable_negative"]
        | stage1_core_classified["is_exploration"]
    )
    stage1_extension_fields = sorted(
        stage1_core_classified.loc[active_mask, "field"].dropna().astype(str).unique().tolist()
    )
    print("Fields eligible for Extended Stage1:", len(stage1_extension_fields))
    print(stage1_extension_fields)

    top_cols = [c for c in (
        "alpha_id", "field", "signal_class", "sharpe", "fitness", "turnover",
        "positions", "score"
    ) if c in stage1_core_classified.columns]
    display(stage1_core_classified.sort_values("score", ascending=False)[top_cols].head(30))


## 5. 第一阶段 — 定向扩展

只有 Core 已经出现潜力的字段才扩展。加入 `120` 以及 `ts_arg_max / ts_arg_min / ts_quantile`，不会对全部字段重新铺一遍。


In [ ]:
# 只对 Core 阶段已出现潜力的字段生成扩展算子和扩展窗口。
# 为潜力字段生成 120 窗口和扩展算子候选。
stage1_extended_candidates = extended_first_order_candidates(
    field_records,
    active_fields=stage1_extension_fields,
    init_decay=INIT_DECAY,
)
stage1_extended_candidates = annotate_candidates(stage1_extended_candidates, TARGET_MODE)
validate_candidate_context(
    stage1_extended_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

print("Stage1 Extended candidates:", len(stage1_extended_candidates))
# 预览扩展候选，并检查已有缓存命中情况。
if stage1_extended_candidates:
    ext_preview = pd.DataFrame(stage1_extended_candidates)
    display(ext_preview[[c for c in (
        "field", "operator", "window", "search_tier",
        "operator_count", "data_field_count", "expr"
    ) if c in ext_preview.columns]].head(30))
    stage1_extended_resume = resume_summary(
        stage1_extended_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )
else:
    stage1_extended_resume = None


In [ ]:
# 只有开启扩展开关且存在候选时才运行 Stage1 Extended 回测。
if RUN_STAGE1_EXTENDED and stage1_extended_candidates:
    stage1_extended_results = simulate_candidates(
        stage1_extended_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage1_extended_results = pd.DataFrame()
    print("Stage1 Extended 未运行。")


## 6. 第一阶段 — 合并与晋级（支持 1→2→3→6 直达）

缓存恢复、Core/Extended 重建和 Stage1 多样性晋级已全部移入 `machine_lib_V2_1.py`。

即使 `field_records` 不在当前 Kernel 中，第 6 步也会按当前 Dataset / Coverage 参数自动重建字段；只读取 `alpha_results.db`，**不会新建 simulation，也不会 POST**。


In [10]:
# 1236 直达：复杂逻辑已经全部放到 machine_lib_V2_1.py。
stage1_bundle = restore_stage1_cache_and_promote(
    session=s if "s" in globals() else None,
    cache_db=CACHE_DB,
    target_mode=TARGET_MODE,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    neutralization=NEUTRALIZATION,
    delay=DELAY,
    truncation=TRUNCATION,
    test_period=TEST_PERIOD,
    init_decay=INIT_DECAY,
    min_data_coverage=MIN_DATA_COVERAGE,
    data_coverage_column=DATA_COVERAGE_COLUMN,
    max_fields=MAX_FIELDS,
    manual_excluded_fields_by_dataset=MANUAL_EXCLUDED_FIELDS_BY_DATASET,
    field_records=globals().get("field_records"),
    core_ts_ops=CORE_TS_OPS,
    stage1_min_abs_sharpe=STAGE1_MIN_ABS_SHARPE,
    stage1_min_abs_fitness=STAGE1_MIN_ABS_FITNESS,
    stage1_exploration_sharpe=STAGE1_EXPLORATION_SHARPE,
    stage1_exploration_fitness=STAGE1_EXPLORATION_FITNESS,
    stage1_min_positions=STAGE1_MIN_POSITIONS,
    stage1_pp_sharpe_gate=STAGE1_PP_SHARPE_GATE,
    stage1_keep_per_field=3,
    stage1_min_quality_ratio=0.80,
)

# 恢复后续 Stage2 会继续使用的变量。
field_records = stage1_bundle["field_records"]
stage1_core_results = stage1_bundle["core_results"]
stage1_core_classified = stage1_bundle["core_classified"]
stage1_extension_fields = stage1_bundle["extension_fields"]
stage1_extended_results = stage1_bundle["extended_results"]
stage1_results = stage1_bundle["stage1_results"]
stage1_promoted = stage1_bundle["stage1_promoted"]
stage1_selected = stage1_bundle["stage1_selected"]

if not stage1_selected.empty:
    display(stage1_selected[[c for c in (
        "alpha_id", "dataset_id", "field", "data_coverage",
        "operator", "window", "vector_op",
        "sharpe", "fitness", "turnover", "positions",
        "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "score"
    ) if c in stage1_selected.columns]].head(60))


Stage1 field source: memory
Stage1 prepared field expressions: 10
Stage1 Core cache recovery: 150 / 150
Stage1 Core cache missing: 0
Stage1 Core status: {'COMPLETE': 142, 'ERROR': 5, 'RUNNING': 3}
Fields eligible for Extended Stage1: 4
['lending_fee_bid_rate', 'rsk60_crowding', 'rsk60_last', 'rsk60_offer']
Stage1 Extended cache recovery: 72 / 72
Stage1 Extended cache missing: 0
Stage1 Extended status: {'COMPLETE': 72}

Stage1 cache restore / promotion summary
Stage1 total results: 222
Stage1 status: {'COMPLETE': 214, 'ERROR': 5, 'RUNNING': 3}
Stage1 promoted: 11


,alpha_id,dataset_id,field,data_coverage,operator,window,vector_op,sharpe,fitness,turnover,positions,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,score
117,58pJ2zoz,risk60,rsk60_last,0.9503,ts_mean,66.0,vec_sum,-1.53,-2.26,0.0302,1654.0,4,1,True,True,0.987950
147,GrGmZvLG,risk60,rsk60_offer,0.9546,ts_mean,66.0,vec_sum,-1.54,-2.26,0.0284,1620.0,4,1,True,True,0.972185
27,6XpMj6zY,risk60,lending_fee_bid_rate,0.9546,ts_mean,66.0,vec_sum,-1.53,-2.25,0.0283,1621.0,4,1,True,True,0.969820
103,qMN7Ve3E,risk60,rsk60_last,0.9503,ts_std_dev,22.0,vec_avg,-1.59,-1.97,0.0794,1625.0,4,1,True,True,0.929505
122,pwNd3OMo,risk60,rsk60_offer,0.9546,zscore,NaN,vec_avg,-1.44,-2.02,0.0634,1534.0,4,1,True,True,0.879392
2,JjGeodZA,risk60,lending_fee_bid_rate,0.9546,zscore,NaN,vec_avg,-1.42,-1.98,0.0652,1536.0,4,1,True,True,0.867342
107,QPG0g8PK,risk60,rsk60_last,0.9503,zscore,NaN,vec_sum,-1.34,-1.91,0.1187,1621.0,4,1,True,True,0.864527
148,xANvJPnp,risk60,rsk60_offer,0.9546,ts_std_dev,22.0,vec_sum,-1.42,-1.93,0.0760,1534.0,4,1,True,True,0.858896
28,ZYEaARa1,risk60,lending_fee_bid_rate,0.9546,ts_std_dev,22.0,vec_sum,-1.41,-1.92,0.0749,1538.0,4,1,True,True,0.856306
180,ZYEgkGmZ,risk60,rsk60_crowding,0.9512,ts_arg_min,66.0,vec_sum,0.80,0.48,0.0893,1623.0,4,1,True,True,0.827027


## 7. 第二阶段 A — 优先分组中性化

`POWER_POOL_ATOM / ATOM / POWER_POOL` 默认只使用 `sector / industry / subindustry` 三个支持分组，避免无必要引入 `cap / close / volume`。先跑 `group_neutralize`。


In [11]:
# 先对 Stage1 晋级 Alpha 生成 group_neutralize 候选，控制搜索空间。
# 优先只生成 group_neutralize 分支。
stage2_neutral_candidates = second_order_candidates(
    stage1_promoted,
    region=REGION,
    group_ops=("group_neutralize",),
    extended_groups=False,
    target_mode=TARGET_MODE,
)
validate_candidate_context(
    stage2_neutral_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

# 展示候选结构，并查看这些回测是否已存在于缓存。
print("Stage2 Neutralize candidates:", len(stage2_neutral_candidates))
if stage2_neutral_candidates:
    s2a = pd.DataFrame(stage2_neutral_candidates)
    display(s2a[[c for c in (
        "field", "group_operator", "group", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "expr"
    ) if c in s2a.columns]].head(30))
    stage2_neutral_resume = resume_summary(
        stage2_neutral_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


Stage2 Neutralize candidates: 33


,field,group_operator,group,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,expr
0,rsk60_last,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(sector))"
1,rsk60_last,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(industry))"
2,rsk60_last,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(subindustry))"
3,rsk60_offer,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(sector))"
4,rsk60_offer,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(industry))"
5,rsk60_offer,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(subindustry))"
6,lending_fee_bid_rate,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(sector))"
7,lending_fee_bid_rate,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(industry))"
8,lending_fee_bid_rate,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(subindustry))"
9,rsk60_last,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_std_dev(winsorize(ts_backfill(vec_avg(rsk60_last), 120), std=4), 22)), densify(sector))"


Resume summary
Candidates: 33
Unique: 33

Completed cache: 33
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 0

Remaining simulations: 0


In [12]:
# 开启对应开关后运行 Stage2 Neutralize 回测，否则保持为空。
if RUN_STAGE2_NEUTRALIZE and stage2_neutral_candidates:
    stage2_neutral_results = simulate_candidates(
        stage2_neutral_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage2_neutral_results = pd.DataFrame()
    print("Stage2 Neutralize 未运行。")


Resume summary
Candidates: 33
Unique: 33

Completed cache: 33
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 0

Remaining simulations: 0
Requested concurrency: 4
Region concurrency limit: 4
Effective concurrency: 4
Polling timeout per worker: 1800s
[001/33] CACHE COMPLETE
[002/33] CACHE COMPLETE
[003/33] CACHE COMPLETE
[004/33] CACHE COMPLETE
[005/33] CACHE COMPLETE
[006/33] CACHE COMPLETE
[007/33] CACHE COMPLETE
[008/33] CACHE COMPLETE
[009/33] CACHE COMPLETE
[010/33] CACHE COMPLETE
[011/33] CACHE COMPLETE
[012/33] CACHE COMPLETE
[013/33] CACHE COMPLETE
[014/33] CACHE COMPLETE
[015/33] CACHE COMPLETE
[016/33] CACHE COMPLETE
[017/33] CACHE COMPLETE
[018/33] CACHE COMPLETE
[019/33] CACHE COMPLETE
[020/33] CACHE COMPLETE
[021/33] CACHE COMPLETE
[022/33] CACHE COMPLETE
[023/33] CACHE COMPLETE
[024/33] CACHE COMPLETE
[025/33] CACHE COMPLETE
[026/33] CACHE COMPLETE
[027/33] CACHE COMPLETE
[028/33] CACHE COMPLETE
[

In [13]:
# 对 Neutralize 结果执行 Stage2 晋级筛选，保留每个字段的优质候选。
# 对 Neutralize 分支进行 Stage2 质量筛选。
stage2_neutral_promoted, stage2_neutral_selected = promote_candidates_for_target(
    stage2_neutral_results,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=2,
    max_total=40,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)
print("Stage2 Neutralize selected:", len(stage2_neutral_selected))
if not stage2_neutral_selected.empty:
    display(stage2_neutral_selected[[c for c in (
        "alpha_id", "field", "group", "sharpe", "fitness", "turnover",
        "positions", "operator_count", "data_field_count", "score"
    ) if c in stage2_neutral_selected.columns]].head(40))


Stage2 Neutralize selected: 6


,alpha_id,field,group,sharpe,fitness,turnover,positions,operator_count,data_field_count,score
0,vRNZO5eQ,rsk60_last,sector,1.53,2.26,0.0302,1655,6,1,0.899242
4,RRm9EQZb,rsk60_offer,industry,1.54,2.26,0.0283,1620,6,1,0.890909
3,pwNxvzoo,rsk60_offer,sector,1.54,2.26,0.0284,1620,6,1,0.886364
6,xANVrqXl,lending_fee_bid_rate,sector,1.53,2.25,0.0283,1621,6,1,0.866667
1,P0GxMn7p,rsk60_last,industry,1.52,2.25,0.0301,1655,6,1,0.858333
7,XgoVRMoz,lending_fee_bid_rate,industry,1.52,2.23,0.0282,1621,6,1,0.833333


## 8. 第二阶段 B — Group Rank 补充

只有 Neutralize 产出的可用候选少于 `STAGE2_FALLBACK_MIN_SELECTED` 时才生成 `group_rank`，保留这个算子但不默认全铺。


In [ ]:
# 当 Neutralize 候选不足时，才启用 group_rank 作为补充搜索。
# 根据 Neutralize 已选数量判断是否需要启用补充分支。
need_group_rank_fallback = len(stage2_neutral_selected) < STAGE2_FALLBACK_MIN_SELECTED

# 只有候选不足时才生成 group_rank，避免默认全铺。
if need_group_rank_fallback:
    stage2_rank_candidates = second_order_candidates(
        stage1_promoted,
        region=REGION,
        group_ops=("group_rank",),
        extended_groups=False,
        target_mode=TARGET_MODE,
    )
    validate_candidate_context(
        stage2_rank_candidates,
        dataset_id=DATASET_ID,
        target_mode=TARGET_MODE,
    )
else:
    stage2_rank_candidates = []

print("Need group_rank fallback:", need_group_rank_fallback)
print("Stage2 Rank candidates:", len(stage2_rank_candidates))
if stage2_rank_candidates:
    stage2_rank_resume = resume_summary(
        stage2_rank_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


In [ ]:
# 仅在确实需要且开关打开时运行 Group Rank 回测。
if RUN_STAGE2_GROUP_RANK_FALLBACK and stage2_rank_candidates:
    stage2_rank_results = simulate_candidates(
        stage2_rank_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage2_rank_results = pd.DataFrame()
    print("Stage2 Group Rank 未运行。")


## 9. 第二阶段 — 合并与最终预筛


In [14]:
# 合并 Stage2 两条分支结果，去重并生成 Repair 前的正式候选。
# 兼容跳过 Group Rank：未运行的分支自动视为空表。
if "stage2_neutral_results" not in globals():
    stage2_neutral_results = pd.DataFrame()
if "stage2_rank_results" not in globals():
    stage2_rank_results = pd.DataFrame()

stage2_frames = [
    x for x in (stage2_neutral_results, stage2_rank_results)
    if isinstance(x, pd.DataFrame) and not x.empty
]
if stage2_frames:
    stage2_results = pd.concat(stage2_frames, ignore_index=True, sort=False)
    if "sim_key" in stage2_results.columns:
        stage2_results = stage2_results.drop_duplicates(subset=["sim_key"], keep="last")
else:
    stage2_results = pd.DataFrame()

# 按目标模式进行最终 Stage2 晋级筛选。
stage2_promoted, stage2_selected = promote_candidates_for_target(
    stage2_results,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=2,
    max_total=40,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)
validate_candidate_context(stage2_promoted, dataset_id=DATASET_ID, target_mode=TARGET_MODE)

print("Stage2 total results:", len(stage2_results))
print("Stage2 selected:", len(stage2_selected))
if not stage2_selected.empty:
    display(stage2_selected[[c for c in (
        "alpha_id", "dataset_id", "field", "group_operator", "group", "sharpe",
        "fitness", "turnover", "positions", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "score"
    ) if c in stage2_selected.columns]].head(40))


Stage2 total results: 33
Stage2 selected: 6


,alpha_id,dataset_id,field,group_operator,group,sharpe,fitness,turnover,positions,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,score
0,vRNZO5eQ,risk60,rsk60_last,group_neutralize,sector,1.53,2.26,0.0302,1655,6,1,True,True,0.899242
4,RRm9EQZb,risk60,rsk60_offer,group_neutralize,industry,1.54,2.26,0.0283,1620,6,1,True,True,0.890909
3,pwNxvzoo,risk60,rsk60_offer,group_neutralize,sector,1.54,2.26,0.0284,1620,6,1,True,True,0.886364
6,xANVrqXl,risk60,lending_fee_bid_rate,group_neutralize,sector,1.53,2.25,0.0283,1621,6,1,True,True,0.866667
1,P0GxMn7p,risk60,rsk60_last,group_neutralize,industry,1.52,2.25,0.0301,1655,6,1,True,True,0.858333
7,XgoVRMoz,risk60,lending_fee_bid_rate,group_neutralize,industry,1.52,2.23,0.0282,1621,6,1,True,True,0.833333


## 10. Regional Repair V2 — 区域鲁棒性候选

针对当前 GLB 候选的 EMEA / APAC / Sub-universe 偏弱问题：每个字段只取 Stage2 最强父 Alpha，默认最多 3 个父信号；每个父信号生成 5 个 Country 方向变体，共约 15 条。

本节只生成候选和查看缓存状态，**不会回测 POST**。


In [15]:
# 动态 reload PY：替换 machine_lib_V2_1.py 后无需重启当前 Kernel。
import importlib
import machine_lib_V2_1 as _ml_v21
_ml_v21 = importlib.reload(_ml_v21)

repair_candidates = _ml_v21.regional_repair_candidates(
    stage2_selected,
    target_mode=TARGET_MODE,
    max_parents=3,
    humps=(0.01, 0.02),
)

validate_candidate_context(
    repair_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

print("Regional Repair V2 candidates:", len(repair_candidates))

if repair_candidates:
    rp = pd.DataFrame(repair_candidates)
    display(rp[[c for c in (
        "regional_parent_alpha_id", "regional_parent_field",
        "repair", "group", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "expr"
    ) if c in rp.columns]].head(30))

    repair_resume = resume_summary(
        repair_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


Regional Repair V2 candidates: 15


,regional_parent_alpha_id,regional_parent_field,repair,group,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,expr
0,vRNZO5eQ,rsk60_last,regional_country_neutralize,country,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country))"
1,vRNZO5eQ,rsk60_last,regional_country_rank,country,6,1,True,True,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country))"
2,vRNZO5eQ,rsk60_last,regional_industry_then_country,industry+country,6,1,True,True,"group_neutralize(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(industry)), densify(c..."
3,vRNZO5eQ,rsk60_last,regional_country_hump_0.01,country,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country)), hump=0.01)"
4,vRNZO5eQ,rsk60_last,regional_country_hump_0.02,country,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country)), hump=0.02)"
5,RRm9EQZb,rsk60_offer,regional_country_neutralize,country,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(country))"
6,RRm9EQZb,rsk60_offer,regional_country_rank,country,6,1,True,True,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(country))"
7,RRm9EQZb,rsk60_offer,regional_industry_then_country,industry+country,6,1,True,True,"group_neutralize(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(industry)), densify(..."
8,RRm9EQZb,rsk60_offer,regional_country_hump_0.01,country,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(country)), hump=0.01)"
9,RRm9EQZb,rsk60_offer,regional_country_hump_0.02,country,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(country)), hump=0.02)"


Resume summary
Candidates: 15
Unique: 15

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 15

Remaining simulations: 15


## 11. Regional Repair V2 — 正式回测

确认第 10 阶段约 15 条候选结构无误后，将 `RUN_REPAIR = True` 再运行本节。回测仍复用 SQLite 缓存与当前并发保护。


In [17]:
# 第10阶段已经生成 repair_candidates；这里仅确认数量。
print("Regional Repair V2 ready:", len(repair_candidates) if "repair_candidates" in globals() else 0)


Regional Repair V2 ready: 15


In [18]:
# 只有 RUN_REPAIR=True 且存在 Regional Repair V2 候选时才正式回测。
if RUN_REPAIR and repair_candidates:
    repair_results = simulate_candidates(
        repair_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    repair_results = pd.DataFrame()
    print("Regional Repair V2 未运行。")


Resume summary
Candidates: 15
Unique: 15

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 15

Remaining simulations: 15
Requested concurrency: 4
Region concurrency limit: 4
Effective concurrency: 4
Polling timeout per worker: 1800s
[001/15] POST submitted
           Location=https://api.worldquantbrain.com/simulations/3XalSq8iJ4Dy94XEm1S1tap
[001/15] INITIAL POLL waiting 5.0s
[002/15] POST submitted
           Location=https://api.worldquantbrain.com/simulations/4qwxpDs251taUo7Tkqyjxv
[002/15] INITIAL POLL waiting 5.0s
[003/15] POST submitted
           Location=https://api.worldquantbrain.com/simulations/3hTClyfsi4SXatqVG959cIp
[003/15] INITIAL POLL waiting 5.0s
[004/15] POST submitted
           Location=https://api.worldquantbrain.com/simulations/3TOdvT50P4FycEXdYYIbAVu
[004/15] INITIAL POLL waiting 5.0s
[001/15] POLL PENDING | waiting 5.0s
[002/15] POLL PENDING | waiting 5.0s
[003/15] PO

In [21]:
# =========================================
# Stage 11 Regional Repair V2 详细结果展示
# 只展示 repair_results，不混入 Stage2 / Stage12
# =========================================

if "repair_results" not in globals() or repair_results is None or repair_results.empty:
    print("❌ repair_results 为空，请确认第11阶段已经运行完成。")

else:
    rows = []

    for _, r in repair_results.iterrows():
        c = r.get("candidate", {})
        if not isinstance(c, dict):
            c = {}

        rows.append({
            # Alpha 基础信息
            "alpha_id": r.get("alpha_id"),
            "status": r.get("status"),
            "cached": r.get("cached"),

            # Repair 来源
            "parent_alpha_id": c.get("regional_parent_alpha_id"),
            "field": c.get("regional_parent_field") or c.get("field"),
            "repair": c.get("repair"),
            "group": c.get("group"),

            # 回测核心指标
            "sharpe": r.get("sharpe"),
            "fitness": r.get("fitness"),
            "turnover": r.get("turnover"),
            "returns": r.get("returns"),
            "margin": r.get("margin"),
            "drawdown": r.get("drawdown"),
            "positions": r.get("positions"),

            # PPL / ATOM 结构
            "operator_count": c.get("operator_count"),
            "data_field_count": c.get("data_field_count"),
            "pp_structure_ok": c.get("pp_structure_ok"),
            "atom_structure_ok": c.get("atom_structure_ok"),

            # 具体表达式
            "expr": c.get("expr"),

            # 调试信息
            "sim_key": r.get("sim_key"),
            "error": r.get("error"),
        })

    stage11_detail = pd.DataFrame(rows)

    # 按字段 → 父Alpha → Sharpe 排序
    stage11_detail = stage11_detail.sort_values(
        ["field", "parent_alpha_id", "sharpe"],
        ascending=[True, True, False],
        na_position="last"
    ).reset_index(drop=True)

    print("Stage 11 Regional Repair V2")
    print("--------------------------------")
    print("总结果数:", len(stage11_detail))
    print("COMPLETE:", (stage11_detail["status"] == "COMPLETE").sum())
    print()

    display(stage11_detail)

Stage 11 Regional Repair V2
--------------------------------
总结果数: 15
COMPLETE: 15



,alpha_id,status,cached,parent_alpha_id,field,repair,group,sharpe,fitness,turnover,returns,margin,drawdown,positions,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,expr,sim_key,error
0,pwNMx6lj,COMPLETE,False,xANVrqXl,lending_fee_bid_rate,regional_country_neutralize,country,1.54,2.22,0.0305,0.2596,0.017030,None,None,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(country))",ed92a9b9e4fd4ceba63e210ee26f69285f92d105031d22f9707074705c4f7986,None
1,YPvE8plv,COMPLETE,False,xANVrqXl,lending_fee_bid_rate,regional_industry_then_country,industry+country,1.53,2.21,0.0310,0.2612,0.016879,None,None,6,1,True,True,"group_neutralize(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(industry)),...",6a0d3d10e49bc5f5ba99a83728dff3ef73d86fbc2f3fc0191330996f849d496b,None
2,E5G8NPjm,COMPLETE,False,xANVrqXl,lending_fee_bid_rate,regional_country_hump_0.02,country,1.43,1.97,0.0204,0.2363,0.023127,None,None,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(country)), hump=0.02)",75d29cf8d95ecc51530252f0651aef8c28fb4790b1d06b2fe032a37cf8d4a03e,None
3,npNemXM3,COMPLETE,False,xANVrqXl,lending_fee_bid_rate,regional_country_hump_0.01,country,1.42,1.97,0.0215,0.2415,0.022451,None,None,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(country)), hump=0.01)",6d58804b190f307c70e6ec75126f8f2437c05b44a0d80504c17aa06f2b806745,None
4,xAN2VxYw,COMPLETE,False,xANVrqXl,lending_fee_bid_rate,regional_country_rank,country,1.03,0.61,0.0385,0.0438,0.002277,None,None,6,1,True,True,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(country))",183f96013434459bfeb970150bad33f06c8a16af8c1055a93ca07fcf249e9ff3,None
5,78zvGaPZ,COMPLETE,False,vRNZO5eQ,rsk60_last,regional_country_neutralize,country,1.51,2.18,0.0323,0.2612,0.016154,None,None,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country))",9bb9ef75a9bb5c3f9213082bcb89ed0c96e768d58982e0e50cf45ee47c1331fc,None
6,RRml5jrn,COMPLETE,False,vRNZO5eQ,rsk60_last,regional_industry_then_country,industry+country,1.50,2.17,0.0328,0.2627,0.016012,None,None,6,1,True,True,"group_neutralize(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(industry)), densify(c...",697143b3645ed8dda7d087ba9204d9b887defb63accc8b72917b3e3a613e1fb6,None
7,blQrxGQN,COMPLETE,False,vRNZO5eQ,rsk60_last,regional_country_hump_0.01,country,1.37,1.89,0.0224,0.2380,0.021205,None,None,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country)), hump=0.01)",350da41dc967332c57a35c26b621523b8baf6d11f4409b4416d4317d7c361ae8,None
8,ZYEdXjq1,COMPLETE,False,vRNZO5eQ,rsk60_last,regional_country_hump_0.02,country,1.37,1.85,0.0210,0.2286,0.021769,None,None,7,1,True,True,"hump(group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country)), hump=0.02)",62a73a4140cc8f5d28b9ec3161ced51b7f50a26435f9e9dab4e917138efa842d,None
9,pwNMWR1q,COMPLETE,False,vRNZO5eQ,rsk60_last,regional_country_rank,country,0.94,0.53,0.0408,0.0399,0.001955,None,None,6,1,True,True,"group_rank(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(country))",5534e22668385bfd14200f4fe6d3031bb896acb9f6b0667eec4b5143549b6f35,None


## 12. 最终候选

最终只保留当前目标模式的本地结构/质量预筛候选；平台 `/check` 仍是最终裁决。


In [19]:
# 合并 Stage2 与 Repair 结果，形成当前目标模式下的最终候选池。
# 收集 Stage2 与 Repair 两部分可用结果。
combined_final_frames = []
if not stage2_selected.empty:
    combined_final_frames.append(stage2_selected)
if not repair_results.empty:
    combined_final_frames.append(repair_results)

if combined_final_frames:
    final_pool = pd.concat(combined_final_frames, ignore_index=True, sort=False)
    if "sim_key" in final_pool.columns:
        final_pool = final_pool.drop_duplicates(subset=["sim_key"], keep="last")
else:
    final_pool = pd.DataFrame()

# 进行最终本地预筛；平台实时 /check 仍是最终裁决。
_final_promoted, final_results = promote_candidates_for_target(
    final_pool,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=4,
    max_total=100,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)

print("Final candidates:", len(final_results))
# 给候选标记本地结构类型，便于区分 PP、ATOM 和普通研究候选。
if not final_results.empty:
    def local_type(c):
        if not isinstance(c, dict):
            return "UNKNOWN"
        if c.get("pp_atom_structure_ok"):
            return "PP + ATOM structure"
        if c.get("pp_structure_ok"):
            return "Power Pool structure"
        if c.get("atom_structure_ok"):
            return "ATOM structure"
        return "Regular / research"

    final_results = final_results.copy()
    final_results["local_type"] = final_results["candidate"].map(local_type)
    display(final_results[[c for c in (
        "alpha_id", "dataset_id", "field", "local_type", "data_coverage",
        "sharpe", "fitness", "turnover", "positions", "operator_count",
        "data_field_count", "pyramid_multiplier", "score"
    ) if c in final_results.columns]].head(100))


Final candidates: 12


,alpha_id,dataset_id,field,local_type,data_coverage,sharpe,fitness,turnover,positions,operator_count,data_field_count,score
1,RRm9EQZb,risk60,rsk60_offer,PP + ATOM structure,0.9546,1.54,2.26,0.0283,1620,6,1,0.796429
2,pwNxvzoo,risk60,rsk60_offer,PP + ATOM structure,0.9546,1.54,2.26,0.0284,1620,6,1,0.789286
0,vRNZO5eQ,risk60,rsk60_last,PP + ATOM structure,0.9503,1.53,2.26,0.0302,1655,6,1,0.783333
11,MPGegQLL,risk60,rsk60_offer,PP + ATOM structure,0.9546,1.55,2.23,0.0306,1624,6,1,0.772619
3,xANVrqXl,risk60,lending_fee_bid_rate,PP + ATOM structure,0.9546,1.53,2.25,0.0283,1621,6,1,0.725000
16,pwNMx6lj,risk60,lending_fee_bid_rate,PP + ATOM structure,0.9546,1.54,2.22,0.0305,1625,6,1,0.721429
4,P0GxMn7p,risk60,rsk60_last,PP + ATOM structure,0.9503,1.52,2.25,0.0301,1655,6,1,0.719048
13,1YplGGM6,risk60,rsk60_offer,PP + ATOM structure,0.9546,1.54,2.22,0.0311,1623,6,1,0.685714
5,XgoVRMoz,risk60,lending_fee_bid_rate,PP + ATOM structure,0.9546,1.52,2.23,0.0282,1621,6,1,0.666667
18,YPvE8plv,risk60,lending_fee_bid_rate,PP + ATOM structure,0.9546,1.53,2.21,0.0310,1624,6,1,0.623810


## 13. 最终提交检查

仍然**不会自动提交 Alpha**。这里把 PASS 数量和失败原因结构化保存。


In [20]:
# 对最终候选再次执行平台实时检查，并统计主要失败原因。
# 开关开启时对最终候选执行实时提交检查。
if RUN_FINAL_CHECK and not final_results.empty:
    final_checks = check_submission_candidates(
        s,
        final_results,
        limit=100,
        cache_db=CACHE_DB,
    )
else:
    final_checks = pd.DataFrame()
    print("Final check 未运行。")

if not final_checks.empty:
    pass_count = int(final_checks["check_pass"].fillna(False).sum())
    print("Final CHECK PASS:", pass_count, "/", len(final_checks))
    display(final_checks[[c for c in (
        "alpha_id", "field", "local_type", "sharpe", "fitness", "turnover",
        "check_pass", "self_correlation", "failed_check_names", "check_error"
    ) if c in final_checks.columns]].head(100))

    # 汇总失败类型，后续可用于优化 Repair 和搜索策略。
    failed_names = (
        final_checks.loc[~final_checks["check_pass"].fillna(False), "failed_check_names"]
        .explode()
        .dropna()
        .astype(str)
        .value_counts()
    )
    if not failed_names.empty:
        print("\nFail reason counts:")
        display(failed_names.rename("count").to_frame())


Final CHECK PASS: 0 / 12


,alpha_id,field,local_type,sharpe,fitness,turnover,check_pass,self_correlation,failed_check_names,check_error
0,RRm9EQZb,rsk60_offer,PP + ATOM structure,1.54,2.26,0.0283,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
1,pwNxvzoo,rsk60_offer,PP + ATOM structure,1.54,2.26,0.0284,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
2,vRNZO5eQ,rsk60_last,PP + ATOM structure,1.53,2.26,0.0302,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
3,MPGegQLL,rsk60_offer,PP + ATOM structure,1.55,2.23,0.0306,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
4,xANVrqXl,lending_fee_bid_rate,PP + ATOM structure,1.53,2.25,0.0283,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
5,pwNMx6lj,lending_fee_bid_rate,PP + ATOM structure,1.54,2.22,0.0305,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
6,P0GxMn7p,rsk60_last,PP + ATOM structure,1.52,2.25,0.0301,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
7,1YplGGM6,rsk60_offer,PP + ATOM structure,1.54,2.22,0.0311,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
8,XgoVRMoz,lending_fee_bid_rate,PP + ATOM structure,1.52,2.23,0.0282,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
9,YPvE8plv,lending_fee_bid_rate,PP + ATOM structure,1.53,2.21,0.0310,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None



Fail reason counts:


,count
LOW_SHARPE,12
LOW_GLB_EMEA_SHARPE,12
LOW_SUB_UNIVERSE_SHARPE,12
LOW_GLB_APAC_SHARPE,7


## 14. 结果导出与上下文汇总

Simulation 仍在原 `alpha_results` 表；V2.1 另外在同一个 DB 的 `alpha_contexts` 表保存 Dataset / Target / Stage 等研究上下文，不改变旧缓存主键。


In [ ]:
# 将各阶段非空结果导出为 CSV，并输出当前 Dataset/Target 的上下文统计。
# 创建本轮结果目录。
OUTPUT_DIR = project_dir / "results_v2_1"
OUTPUT_DIR.mkdir(exist_ok=True)

exports = {
    "data_fields_filtered.csv": df_fields,
    "stage1_core_results.csv": stage1_core_results,
    "stage1_extended_results.csv": stage1_extended_results,
    "stage1_results.csv": stage1_results,
    "stage2_neutral_results.csv": stage2_neutral_results,
    "stage2_rank_results.csv": stage2_rank_results,
    "stage2_results.csv": stage2_results,
    "pre_repair_checks.csv": pre_repair_checks,
    "repair_results.csv": repair_results,
    "final_results.csv": final_results,
    "final_checks.csv": final_checks,
}

# 只导出非空结果表，避免生成无意义的空文件。
for filename, frame in exports.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        path = OUTPUT_DIR / filename
        frame.to_csv(path, index=False)
        print("Exported:", path)

print("\nCurrent Dataset / Target contexts:")
display(context_summary(CACHE_DB, dataset_id=DATASET_ID, target_mode=TARGET_MODE))


## 15. 缓存与维护

全局 simulation cache 与当前研究 context 分开看，避免把不同 Dataset 的总数误认为当前 Run。


In [ ]:
# 分开查看全局回测缓存和当前 Dataset/Target 的研究上下文。
print("Global simulation cache:")
cache_summary(CACHE_DB)

print("\nCurrent Dataset / Target contexts:")
display(context_summary(CACHE_DB, dataset_id=DATASET_ID, target_mode=TARGET_MODE))


In [22]:
# ============================================================
# Alpha /check 详细指标展示
# 直接解析 Overall / AMER / EMEA / APAC / Sub-universe / 2Y
# 只 GET 检查，不提交 Alpha
# ============================================================

import pandas as pd
import importlib
import machine_lib_V2_1 as ml

ml = importlib.reload(ml)

# 如果当前 Kernel 还没有登录 Session，则自动登录
if "s" not in globals() or s is None:
    s = ml.login()


# ============================================================
# ① 在这里填写你要检查的 Alpha ID
# ============================================================

ALPHA_IDS = [
    "MPGegQLL",
    "pwNMx6lj",
]

# 以后想检查别的，直接改成：
# ALPHA_IDS = [
#     "xxxxxxx",
#     "yyyyyyy",
# ]


# ============================================================
# ② 辅助函数
# ============================================================

def _safe_round(x, n=4):
    try:
        if x is None:
            return None
        return round(float(x), n)
    except Exception:
        return x


def _get_check(check_map, name):
    """
    返回:
        value
        cutoff
        result
    """
    c = check_map.get(name)

    if not isinstance(c, dict):
        return None, None, None

    return (
        c.get("value"),
        c.get("limit"),
        c.get("result"),
    )


# ============================================================
# ③ 调用 BRAIN /check
# ============================================================

rows = []

for i, alpha_id in enumerate(ALPHA_IDS, 1):

    print(f"[{i}/{len(ALPHA_IDS)}] checking {alpha_id} ...")

    try:
        detail = ml.get_check_submission_detailed(
            s,
            str(alpha_id)
        )

        checks = detail.get("checks", []) or []

        # name -> check detail
        check_map = {
            str(c.get("name")): c
            for c in checks
            if isinstance(c, dict) and c.get("name")
        }

        # ----------------------------------------------------
        # 核心指标
        # ----------------------------------------------------

        sharpe, sharpe_cut, sharpe_result = _get_check(
            check_map,
            "LOW_SHARPE"
        )

        fitness, fitness_cut, fitness_result = _get_check(
            check_map,
            "LOW_FITNESS"
        )

        amer, amer_cut, amer_result = _get_check(
            check_map,
            "LOW_GLB_AMER_SHARPE"
        )

        emea, emea_cut, emea_result = _get_check(
            check_map,
            "LOW_GLB_EMEA_SHARPE"
        )

        apac, apac_cut, apac_result = _get_check(
            check_map,
            "LOW_GLB_APAC_SHARPE"
        )

        sub, sub_cut, sub_result = _get_check(
            check_map,
            "LOW_SUB_UNIVERSE_SHARPE"
        )

        y2, y2_cut, y2_result = _get_check(
            check_map,
            "LOW_2Y_SHARPE"
        )

        low_turn, low_turn_cut, low_turn_result = _get_check(
            check_map,
            "LOW_TURNOVER"
        )

        high_turn, high_turn_cut, high_turn_result = _get_check(
            check_map,
            "HIGH_TURNOVER"
        )

        # 所有 FAIL
        failed_checks = [
            str(c.get("name"))
            for c in checks
            if c.get("result") == "FAIL"
        ]

        rows.append({

            "alpha_id": alpha_id,

            # Overall
            "Sharpe": _safe_round(sharpe),
            "Sharpe_cutoff": _safe_round(sharpe_cut),
            "Sharpe_status": sharpe_result,

            "Fitness": _safe_round(fitness),
            "Fitness_cutoff": _safe_round(fitness_cut),
            "Fitness_status": fitness_result,

            # GLB 三大区域
            "AMER": _safe_round(amer),
            "AMER_cutoff": _safe_round(amer_cut),
            "AMER_status": amer_result,

            "EMEA": _safe_round(emea),
            "EMEA_cutoff": _safe_round(emea_cut),
            "EMEA_status": emea_result,

            "APAC": _safe_round(apac),
            "APAC_cutoff": _safe_round(apac_cut),
            "APAC_status": apac_result,

            # Sub-universe
            "SubUniverse": _safe_round(sub),
            "Sub_cutoff": _safe_round(sub_cut),
            "Sub_status": sub_result,

            # 2Y
            "2Y_Sharpe": _safe_round(y2),
            "2Y_cutoff": _safe_round(y2_cut),
            "2Y_status": y2_result,

            # Turnover
            "LowTurnover_value": _safe_round(low_turn),
            "LowTurnover_cutoff": _safe_round(low_turn_cut),
            "LowTurnover_status": low_turn_result,

            "HighTurnover_value": _safe_round(high_turn),
            "HighTurnover_cutoff": _safe_round(high_turn_cut),
            "HighTurnover_status": high_turn_result,

            # 总体
            "check_pass": bool(detail.get("passed")),
            "self_correlation": detail.get("self_correlation"),

            "FAIL_count": len(failed_checks),
            "FAIL_checks": failed_checks,

            "error": None,
        })

    except Exception as e:

        rows.append({
            "alpha_id": alpha_id,
            "check_pass": False,
            "FAIL_count": None,
            "FAIL_checks": None,
            "error": f"{type(e).__name__}: {e}",
        })


# ============================================================
# ④ 完整结果
# ============================================================

alpha_check_detail = pd.DataFrame(rows)

print()
print("=" * 80)
print("Alpha /check 完整结果")
print("=" * 80)

display(alpha_check_detail)


# ============================================================
# ⑤ 最重要的核心对比表
# ============================================================

core_cols = [
    "alpha_id",

    "Sharpe",
    "Sharpe_cutoff",
    "Sharpe_status",

    "AMER",
    "AMER_cutoff",
    "AMER_status",

    "EMEA",
    "EMEA_cutoff",
    "EMEA_status",

    "APAC",
    "APAC_cutoff",
    "APAC_status",

    "SubUniverse",
    "Sub_cutoff",
    "Sub_status",

    "2Y_Sharpe",
    "2Y_cutoff",
    "2Y_status",

    "check_pass",
    "FAIL_count",
    "FAIL_checks",
]

core_cols = [
    c for c in core_cols
    if c in alpha_check_detail.columns
]

alpha_check_core = alpha_check_detail[core_cols].copy()

print()
print("=" * 80)
print("核心提交指标")
print("=" * 80)

display(alpha_check_core)

[1/2] checking MPGegQLL ...
[2/2] checking pwNMx6lj ...

Alpha /check 完整结果


,alpha_id,Sharpe,Sharpe_cutoff,Sharpe_status,Fitness,Fitness_cutoff,Fitness_status,AMER,AMER_cutoff,AMER_status,EMEA,EMEA_cutoff,EMEA_status,APAC,APAC_cutoff,APAC_status,SubUniverse,Sub_cutoff,Sub_status,2Y_Sharpe,2Y_cutoff,2Y_status,LowTurnover_value,LowTurnover_cutoff,LowTurnover_status,HighTurnover_value,HighTurnover_cutoff,HighTurnover_status,check_pass,self_correlation,FAIL_count,FAIL_checks,error
0,MPGegQLL,1.55,1.58,FAIL,2.23,1.0,PASS,1.22,1.0,PASS,0.41,1.0,FAIL,1.12,1.0,PASS,0.88,0.95,FAIL,2.34,1.58,PASS,0.0306,0.01,PASS,0.0306,0.7,PASS,False,None,3,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
1,pwNMx6lj,1.54,1.58,FAIL,2.22,1.0,PASS,1.22,1.0,PASS,0.42,1.0,FAIL,1.14,1.0,PASS,0.88,0.94,FAIL,2.35,1.58,PASS,0.0305,0.01,PASS,0.0305,0.7,PASS,False,None,3,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None



核心提交指标


,alpha_id,Sharpe,Sharpe_cutoff,Sharpe_status,AMER,AMER_cutoff,AMER_status,EMEA,EMEA_cutoff,EMEA_status,APAC,APAC_cutoff,APAC_status,SubUniverse,Sub_cutoff,Sub_status,2Y_Sharpe,2Y_cutoff,2Y_status,check_pass,FAIL_count,FAIL_checks
0,MPGegQLL,1.55,1.58,FAIL,1.22,1.0,PASS,0.41,1.0,FAIL,1.12,1.0,PASS,0.88,0.95,FAIL,2.34,1.58,PASS,False,3,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]"
1,pwNMx6lj,1.54,1.58,FAIL,1.22,1.0,PASS,0.42,1.0,FAIL,1.14,1.0,PASS,0.88,0.94,FAIL,2.35,1.58,PASS,False,3,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_SUB_UNIVERSE_SHARPE]"


In [42]:
# ============================================================
# PPL VWAP60 — Sharpe Recovery Scan
#
# 已知：
# 原始反向信号：
#   -mean_vwap_return_60m_pre_close_2
#
# 原始信号约：
#   Sharpe ≈ 1.38
#   Turnover ≈ 140%
#
# 已验证：
# ts_target_tvr_decay(
#     -field,
#     lambda_min=0,
#     lambda_max=1,
#     target_tvr=0.55
# )
#
# 结果：
#   Sharpe   = 0.74
#   Turnover = 66.46%
#
# 本轮目标：
#   20% <= Turnover <= 70%
#   Sharpe >= 1.0
#
# 策略：
#   A. 微调 lambda 范围
#   B. 微调 target_tvr
#   C. 用简单 ts_mean 做对照
# ============================================================

import pandas as pd
import importlib
import machine_lib_V2_1 as ml

ml = importlib.reload(ml)

if "s" not in globals() or s is None:
    s = ml.login()


# ============================================================
# 1. 固定环境
# ============================================================

REGION_TEST = "GLB"
UNIVERSE_TEST = "TOPDIV3000"
DELAY_TEST = 1
NEUT_TEST = "SUBINDUSTRY"

DATASET_TEST = "intraday_pv_feats"

FIELD = "mean_vwap_return_60m_pre_close_2"

BASE_SIGNAL = f"-{FIELD}"

SIM_DECAY = 0


# ============================================================
# 2. 候选表达式
# ============================================================

specs = []


def add_spec(name, expr, method, **kwargs):

    row = {
        "name": name,
        "expr": expr,
        "method": method,
    }

    row.update(kwargs)

    specs.append(row)


# ------------------------------------------------------------
# A. 已知基准
# 应命中缓存，不重新消耗 simulation
# ------------------------------------------------------------

add_spec(
    "tvr_0_1_target55",
    (
        f"ts_target_tvr_decay("
        f"{BASE_SIGNAL}, "
        f"lambda_min=0, "
        f"lambda_max=1, "
        f"target_tvr=0.55)"
    ),
    "target_tvr",
    lambda_min=0,
    lambda_max=1,
    target_tvr=0.55,
)


# ------------------------------------------------------------
# B. 固定 target=0.55
# 调整 lambda 范围
# ------------------------------------------------------------

lambda_specs = [
    (0.05, 0.95),
    (0.10, 0.90),
    (0.10, 0.70),
    (0.10, 0.50),
]


for lmin, lmax in lambda_specs:

    add_spec(
        (
            f"tvr_lambda_"
            f"{str(lmin).replace('.', '')}_"
            f"{str(lmax).replace('.', '')}"
        ),
        (
            f"ts_target_tvr_decay("
            f"{BASE_SIGNAL}, "
            f"lambda_min={lmin}, "
            f"lambda_max={lmax}, "
            f"target_tvr=0.55)"
        ),
        "target_tvr",
        lambda_min=lmin,
        lambda_max=lmax,
        target_tvr=0.55,
    )


# ------------------------------------------------------------
# C. 固定 lambda=0.10~0.90
# 微调 Target TVR
# ------------------------------------------------------------

for target in [
    0.50,
    0.60,
]:

    add_spec(
        f"tvr_01_09_target{int(target*100)}",
        (
            f"ts_target_tvr_decay("
            f"{BASE_SIGNAL}, "
            f"lambda_min=0.1, "
            f"lambda_max=0.9, "
            f"target_tvr={target})"
        ),
        "target_tvr",
        lambda_min=0.1,
        lambda_max=0.9,
        target_tvr=target,
    )


# ------------------------------------------------------------
# D. 简单平滑对照
#
# Forum经验：
# Turnover过高时，ts_mean也是常见处理办法。
#
# 看能否比 Target TVR 更好地保存 Sharpe。
# ------------------------------------------------------------

for window in [
    2,
    3,
    4,
    5,
]:

    add_spec(
        f"ts_mean_{window}",
        f"ts_mean({BASE_SIGNAL}, {window})",
        "ts_mean",
        window=window,
    )


print("候选数:", len(specs))


# ============================================================
# 3. Candidate
# ============================================================

candidates = []


for spec in specs:

    c = {

        "dataset_id":
            DATASET_TEST,

        "field":
            FIELD,

        "field_type":
            "MATRIX",

        "vector_op":
            None,

        "stage":
            3,

        "search_tier":
            "ppl_vwap60_sharpe_recovery",

        "operator":
            spec["method"],

        "window":
            spec.get("window"),

        "decay":
            SIM_DECAY,

        "expr":
            spec["expr"],

        "target_mode":
            "POWER_POOL",

        "repair":
            spec["name"],

        "lambda_min":
            spec.get("lambda_min"),

        "lambda_max":
            spec.get("lambda_max"),

        "target_tvr":
            spec.get("target_tvr"),
    }


    try:
        c = ml.annotate_candidate_strategy(
            c,
            "POWER_POOL"
        )
    except Exception:
        pass


    candidates.append(c)


# ============================================================
# 4. Preview
# ============================================================

preview = pd.DataFrame([{

    "repair":
        c.get("repair"),

    "operator":
        c.get("operator"),

    "window":
        c.get("window"),

    "lambda_min":
        c.get("lambda_min"),

    "lambda_max":
        c.get("lambda_max"),

    "target_tvr":
        c.get("target_tvr"),

    "operator_count":
        c.get("operator_count"),

    "data_field_count":
        c.get("data_field_count"),

    "expr":
        c.get("expr"),

} for c in candidates])


display(preview)


# ============================================================
# 5. Guard
# ============================================================

ml.validate_candidate_context(
    candidates,
    dataset_id=DATASET_TEST,
    target_mode="POWER_POOL",
)


# ============================================================
# 6. Resume
# ============================================================

ml.resume_summary(

    candidates,

    neutralization=
        NEUT_TEST,

    region=
        REGION_TEST,

    universe=
        UNIVERSE_TEST,

    cache_db=
        CACHE_DB,

    delay=
        DELAY_TEST,

    truncation=
        TRUNCATION,

    test_period=
        TEST_PERIOD,
)


# ============================================================
# 7. Simulation
# ============================================================

vwap60_recovery_results = ml.simulate_candidates(

    candidates,

    neutralization=
        NEUT_TEST,

    region=
        REGION_TEST,

    universe=
        UNIVERSE_TEST,

    session=
        s,

    cache_db=
        CACHE_DB,

    concurrency=
        min(CONCURRENCY, 4),

    delay=
        DELAY_TEST,

    truncation=
        TRUNCATION,

    test_period=
        TEST_PERIOD,

    progress_every=
        3,

    poll_timeout_seconds=
        POLL_TIMEOUT_SECONDS,

    glb_max_concurrency=
        GLB_MAX_CONCURRENCY,

    other_max_concurrency=
        OTHER_MAX_CONCURRENCY,
)


# ============================================================
# 8. 整理
# ============================================================

rows = []


for _, r in vwap60_recovery_results.iterrows():

    c = r.get("candidate", {})

    if not isinstance(c, dict):
        c = {}


    rows.append({

        "alpha_id":
            r.get("alpha_id"),

        "repair":
            c.get("repair"),

        "method":
            c.get("operator"),

        "window":
            c.get("window"),

        "lambda_min":
            c.get("lambda_min"),

        "lambda_max":
            c.get("lambda_max"),

        "target_tvr":
            c.get("target_tvr"),

        "sharpe":
            r.get("sharpe"),

        "fitness":
            r.get("fitness"),

        "turnover":
            r.get("turnover"),

        "returns":
            r.get("returns"),

        "margin":
            r.get("margin"),

        "operator_count":
            c.get("operator_count"),

        "data_field_count":
            c.get("data_field_count"),

        "status":
            r.get("status"),

        "error":
            r.get("error"),
    })


vwap60_recovery_table = pd.DataFrame(rows)


# ============================================================
# 9. 数字化
# ============================================================

for col in [
    "window",
    "lambda_min",
    "lambda_max",
    "target_tvr",
    "sharpe",
    "fitness",
    "turnover",
    "returns",
    "margin",
]:

    if col in vwap60_recovery_table.columns:

        vwap60_recovery_table[col] = pd.to_numeric(
            vwap60_recovery_table[col],
            errors="coerce"
        )


# ============================================================
# 10. 当前 PPL 本地 Gate
# ============================================================

vwap60_recovery_table[
    "sharpe_pass"
] = (
    vwap60_recovery_table["sharpe"]
    >= 1.0
)


vwap60_recovery_table[
    "turnover_pass"
] = (
    (
        vwap60_recovery_table["turnover"]
        >= 0.20
    )
    &
    (
        vwap60_recovery_table["turnover"]
        <= 0.70
    )
)


vwap60_recovery_table[
    "local_ppl_gate"
] = (
    vwap60_recovery_table["sharpe_pass"]
    &
    vwap60_recovery_table["turnover_pass"]
)


# ============================================================
# 11. 排名
#
# 第一优先：
#   同时通过 Sharpe + Turnover
#
# 第二优先：
#   Sharpe
# ============================================================

vwap60_recovery_table = (

    vwap60_recovery_table

    .sort_values(
        [
            "local_ppl_gate",
            "sharpe",
            "turnover",
        ],
        ascending=[
            False,
            False,
            False,
        ],
        na_position="last"
    )

    .reset_index(drop=True)
)


print()
print("=" * 140)
print("VWAP60 Sharpe Recovery — 全部结果")
print("=" * 140)

display(vwap60_recovery_table)


# ============================================================
# 12. 晋级
# ============================================================

vwap60_ppl_shortlist = (

    vwap60_recovery_table[
        vwap60_recovery_table[
            "local_ppl_gate"
        ]
    ]

    .copy()

    .reset_index(drop=True)
)


print()
print("=" * 140)
print(
    "晋级 PPL /check："
    "Sharpe >= 1 且 20% <= Turnover <= 70%"
)
print("=" * 140)


if vwap60_ppl_shortlist.empty:

    print(
        "暂无候选同时通过 Sharpe + Turnover。"
    )

else:

    display(vwap60_ppl_shortlist)

候选数: 11


,repair,operator,window,lambda_min,lambda_max,target_tvr,operator_count,data_field_count,expr
0,tvr_0_1_target55,target_tvr,NaN,0.00,1.00,0.55,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0, lambda_max=1, target_tvr=0.55)"
1,tvr_lambda_005_095,target_tvr,NaN,0.05,0.95,0.55,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.05, lambda_max=0.95, target_tvr=0.55)"
2,tvr_lambda_01_09,target_tvr,NaN,0.10,0.90,0.55,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.1, lambda_max=0.9, target_tvr=0.55)"
3,tvr_lambda_01_07,target_tvr,NaN,0.10,0.70,0.55,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.1, lambda_max=0.7, target_tvr=0.55)"
4,tvr_lambda_01_05,target_tvr,NaN,0.10,0.50,0.55,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.1, lambda_max=0.5, target_tvr=0.55)"
5,tvr_01_09_target50,target_tvr,NaN,0.10,0.90,0.50,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.1, lambda_max=0.9, target_tvr=0.5)"
6,tvr_01_09_target60,target_tvr,NaN,0.10,0.90,0.60,1,1,"ts_target_tvr_decay(-mean_vwap_return_60m_pre_close_2, lambda_min=0.1, lambda_max=0.9, target_tvr=0.6)"
7,ts_mean_2,ts_mean,2.0,NaN,NaN,NaN,1,1,"ts_mean(-mean_vwap_return_60m_pre_close_2, 2)"
8,ts_mean_3,ts_mean,3.0,NaN,NaN,NaN,1,1,"ts_mean(-mean_vwap_return_60m_pre_close_2, 3)"
9,ts_mean_4,ts_mean,4.0,NaN,NaN,NaN,1,1,"ts_mean(-mean_vwap_return_60m_pre_close_2, 4)"


Resume summary
Candidates: 11
Unique: 11

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 11

Remaining simulations: 11
Resume summary
Candidates: 11
Unique: 11

Completed cache: 0
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 11

Remaining simulations: 11
Requested concurrency: 4
Region concurrency limit: 4
Effective concurrency: 4
Polling timeout per worker: 1800s
[001/11] POST submitted
           Location=https://api.worldquantbrain.com/simulations/2VZ9DDge84rfb5jcCPt9xR3
[001/11] INITIAL POLL waiting 5.0s
[004/11] POST submitted
           Location=https://api.worldquantbrain.com/simulations/11ufgZ3iF4p3culLkKFmG75
[004/11] INITIAL POLL waiting 5.0s
[002/11] POST submitted
           Location=https://api.worldquantbrain.com/simulations/19rMRx4DN5k58EP1bEbiVgAu
[002/11] INITIAL POLL waiting 5.0s
[003/11] POST 

,alpha_id,repair,method,window,lambda_min,lambda_max,target_tvr,sharpe,fitness,turnover,returns,margin,operator_count,data_field_count,status,error,sharpe_pass,turnover_pass,local_ppl_gate
0,78z2XJWZ,ts_mean_4,ts_mean,4.0,NaN,NaN,NaN,1.27,0.58,0.6858,0.1416,0.000413,1,1,COMPLETE,None,True,True,True
1,WjAxWeoG,ts_mean_5,ts_mean,5.0,NaN,NaN,NaN,1.17,0.54,0.6080,0.1295,0.000426,1,1,COMPLETE,None,True,True,True
2,9qp26G6x,ts_mean_3,ts_mean,3.0,NaN,NaN,NaN,1.45,0.66,0.7980,0.1652,0.000414,1,1,COMPLETE,None,True,False,False
3,1Yp2mZ0J,ts_mean_2,ts_mean,2.0,NaN,NaN,NaN,1.27,0.49,0.9879,0.1452,0.000294,1,1,COMPLETE,None,True,False,False
4,6Xp2g8q7,tvr_0_1_target55,target_tvr,NaN,0.00,1.00,0.55,0.74,0.37,0.6646,0.1692,0.000509,1,1,COMPLETE,None,False,True,False
5,A1G2aN2R,tvr_01_09_target60,target_tvr,NaN,0.10,0.90,0.60,0.65,0.30,0.7152,0.1478,0.000413,1,1,COMPLETE,None,False,False,False
6,O0G2maYR,tvr_lambda_01_05,target_tvr,NaN,0.10,0.50,0.55,0.59,0.27,0.6628,0.1360,0.000410,1,1,COMPLETE,None,False,True,False
7,0mpad3R6,tvr_lambda_01_07,target_tvr,NaN,0.10,0.70,0.55,0.55,0.24,0.6757,0.1267,0.000375,1,1,COMPLETE,None,False,True,False
8,1Yp2LPe6,tvr_lambda_01_09,target_tvr,NaN,0.10,0.90,0.55,0.54,0.23,0.6780,0.1248,0.000368,1,1,COMPLETE,None,False,True,False
9,ZYE1xMqj,tvr_lambda_005_095,target_tvr,NaN,0.05,0.95,0.55,0.53,0.23,0.6779,0.1254,0.000370,1,1,COMPLETE,None,False,True,False



晋级 PPL /check：Sharpe >= 1 且 20% <= Turnover <= 70%


,alpha_id,repair,method,window,lambda_min,lambda_max,target_tvr,sharpe,fitness,turnover,returns,margin,operator_count,data_field_count,status,error,sharpe_pass,turnover_pass,local_ppl_gate
0,78z2XJWZ,ts_mean_4,ts_mean,4.0,NaN,NaN,NaN,1.27,0.58,0.6858,0.1416,0.000413,1,1,COMPLETE,None,True,True,True
1,WjAxWeoG,ts_mean_5,ts_mean,5.0,NaN,NaN,NaN,1.17,0.54,0.6080,0.1295,0.000426,1,1,COMPLETE,None,True,True,True


In [43]:
# ============================================================
# PPL Final Check — Candidate 78z2XJWZ
#
# 当前候选：
#   Alpha ID   = 78z2XJWZ
#   Field      = mean_vwap_return_60m_pre_close_2
#   Expression = ts_mean(-field, 4)
#   Sharpe     = 1.27
#   Turnover   = 68.58%
#
# 本 Cell：
#   1. 写入符合官方格式的 Description
#   2. 添加 PowerPoolSelected
#   3. 不提交 Alpha
#   4. 轮询 /check
#   5. 输出全部 PPL 检查
# ============================================================

import time
import pandas as pd
import importlib
import machine_lib_V2_1 as ml

ml = importlib.reload(ml)

if "s" not in globals() or s is None:
    s = ml.login()


# ============================================================
# 1. Candidate
# ============================================================

ALPHA_ID = "78z2XJWZ"


# ============================================================
# 2. Power Pool Description
#
# 不包含 Alpha expression。
#
# Idea:
#   利用收盘前60分钟 VWAP return 的短期反转。
#
# Data:
#   该字段描述本身就是：
#   收盘前60分钟 VWAP 的平均收益。
#
# Operator:
#   4日 ts_mean 用于聚合近期行为，并降低原始高换手。
# ============================================================

DESCRIPTION = """
Idea: Stocks with unusually weak late-session VWAP returns may experience short-term price reversal, while stocks with unusually strong late-session performance may subsequently underperform on a relative basis.

Rationale for data used: The mean_vwap_return_60m_pre_close_2 field measures the average return of the volume-weighted average price during the final 60 minutes before market close. This captures recent late-session trading behavior and price pressure.

Rationale for operators used: The signal direction is reversed to capture short-term mean reversion. A four-day time-series mean aggregates the recent late-session return information, reduces day-to-day noise, and brings turnover into a range suitable for the Power Pool high-turnover theme.
""".strip()


print("Description chars:", len(DESCRIPTION))
print()
print(DESCRIPTION)


if len(DESCRIPTION) < 100:
    raise RuntimeError("Description 少于100字符。")


# ============================================================
# 3. 写入 Description + PowerPoolSelected
#
# 注意：
#   这是 PATCH Properties
#   不是 Submit
# ============================================================

print()
print("=" * 110)
print("写入 Power Pool Properties")
print("=" * 110)


property_result = ml.prepare_power_pool_alpha_properties(
    s,
    ALPHA_ID,
    description=DESCRIPTION,
)


display(
    pd.DataFrame([property_result])
)


# ============================================================
# 4. 检查 Properties 是否成功
# ============================================================

details = ml.get_alpha_details(
    s,
    ALPHA_ID
)

print()
print("Current tags:")
print(details.get("tags"))

print()
print(
    "Current description chars:",
    len(
        str(
            (details.get("regular") or {}).get(
                "description",
                ""
            )
        )
    )
)


# ============================================================
# 5. /check 轮询
#
# PPL相关检查可能不是第一次 GET 就全部生成。
#
# 所以：
#   PENDING -> 继续等
# ============================================================

MAX_WAIT = 300
POLL_SECONDS = 8

start = time.time()

last_detail = None


while True:

    try:

        last_detail = ml.get_check_submission_detailed(
            s,
            ALPHA_ID
        )

        checks = (
            last_detail.get("checks", [])
            or []
        )


        pending_checks = [
            c
            for c in checks
            if isinstance(c, dict)
            and str(
                c.get("result") or ""
            ).upper() == "PENDING"
        ]


        print()
        print(
            "checks:",
            len(checks),
            "| pending:",
            len(pending_checks)
        )


        # ----------------------------------------------------
        # 打印当前 PENDING 名称
        # ----------------------------------------------------

        if pending_checks:

            print(
                "PENDING:",
                [
                    x.get("name")
                    for x in pending_checks
                ]
            )


        # ----------------------------------------------------
        # 没有 PENDING 就结束
        # ----------------------------------------------------

        if checks and not pending_checks:
            break


    except Exception as e:

        print(
            "check error:",
            type(e).__name__,
            e
        )


    if time.time() - start >= MAX_WAIT:

        print()
        print("⚠ 达到最大等待时间。")
        break


    time.sleep(POLL_SECONDS)


# ============================================================
# 6. 输出全部 Check
# ============================================================

check_rows = []


if last_detail:

    checks = (
        last_detail.get("checks", [])
        or []
    )


    for c in checks:

        if not isinstance(c, dict):
            continue


        check_rows.append({

            "name":
                c.get("name"),

            "result":
                c.get("result"),

            "value":
                c.get("value"),

            "limit":
                c.get("limit"),

            "description":
                c.get("description"),
        })


ppl_final_check = pd.DataFrame(
    check_rows
)


print()
print("=" * 130)
print("78z2XJWZ — 全部 /check")
print("=" * 130)

display(
    ppl_final_check
)


# ============================================================
# 7. 单独提取我们最关心的检查
# ============================================================

KEYWORDS = [
    "SHARPE",
    "TURNOVER",
    "SUB",
    "HIGH",
    "POWER",
    "THEME",
    "CORRELATION",
]


if not ppl_final_check.empty:

    important_mask = (
        ppl_final_check["name"]
        .fillna("")
        .astype(str)
        .str.upper()
        .apply(
            lambda x:
            any(k in x for k in KEYWORDS)
        )
    )


    ppl_key_checks = (
        ppl_final_check[
            important_mask
        ]
        .copy()
        .reset_index(drop=True)
    )


    print()
    print("=" * 130)
    print("PPL 关键检查")
    print("=" * 130)

    display(
        ppl_key_checks
    )


# ============================================================
# 8. FAIL / WARNING
# ============================================================

if not ppl_final_check.empty:

    ppl_fail_checks = (

        ppl_final_check[
            ppl_final_check["result"]
            .fillna("")
            .astype(str)
            .str.upper()
            .isin([
                "FAIL",
                "WARNING",
            ])
        ]

        .copy()

        .reset_index(drop=True)
    )


    print()
    print("=" * 130)
    print("当前 FAIL / WARNING")
    print("=" * 130)


    if ppl_fail_checks.empty:

        print("✅ 当前没有 FAIL / WARNING")

    else:

        display(
            ppl_fail_checks
        )

Description chars: 758

Idea: Stocks with unusually weak late-session VWAP returns may experience short-term price reversal, while stocks with unusually strong late-session performance may subsequently underperform on a relative basis.

Rationale for data used: The mean_vwap_return_60m_pre_close_2 field measures the average return of the volume-weighted average price during the final 60 minutes before market close. This captures recent late-session trading behavior and price pressure.

Rationale for operators used: The signal direction is reversed to capture short-term mean reversion. A four-day time-series mean aggregates the recent late-session return information, reduces day-to-day noise, and brings turnover into a range suitable for the Power Pool high-turnover theme.

写入 Power Pool Properties


,alpha_id,tag,tags_after,description_chars,properties_applied,http_status
0,78z2XJWZ,PowerPoolSelected,[PowerPoolSelected],758,True,200



Current tags:
['PowerPoolSelected']

Current description chars: 758
check error: JSONDecodeError Expecting value: line 1 column 1 (char 0)
check error: JSONDecodeError Expecting value: line 1 column 1 (char 0)

checks: 29 | pending: 0

78z2XJWZ — 全部 /check


,name,result,value,limit,description
0,LOW_SHARPE,WARNING,1.27,1.58,None
1,CLUSTER_TEST,WARNING,0.71,1.58,None
2,LOW_FITNESS,WARNING,0.58,1.0,None
3,LOW_GLB_AMER_SHARPE,WARNING,0.54,1,None
4,LOW_GLB_EMEA_SHARPE,WARNING,0.33,1,None
5,LOW_GLB_APAC_SHARPE,PASS,1.25,1,None
6,LOW_TURNOVER,PASS,0.6858,0.01,None
7,HIGH_TURNOVER,PASS,0.6858,0.7,None
8,CONCENTRATED_WEIGHT,PASS,None,None,None
9,LOW_SUB_UNIVERSE_SHARPE,PASS,1.19,0.78,None



PPL 关键检查


,name,result,value,limit,description
0,LOW_SHARPE,WARNING,1.27,1.58,None
1,LOW_GLB_AMER_SHARPE,WARNING,0.54,1,None
2,LOW_GLB_EMEA_SHARPE,WARNING,0.33,1,None
3,LOW_GLB_APAC_SHARPE,PASS,1.25,1,None
4,LOW_TURNOVER,PASS,0.6858,0.01,None
5,HIGH_TURNOVER,PASS,0.6858,0.7,None
6,LOW_SUB_UNIVERSE_SHARPE,PASS,1.19,0.78,None
7,SELF_CORRELATION,PASS,None,0.7,None
8,PROD_CORRELATION,PASS,0.5355,0.7,None
9,REGULAR_SUBMISSION,PASS,0,4,None



当前 FAIL / WARNING


,name,result,value,limit,description
0,LOW_SHARPE,WARNING,1.27,1.58,None
1,CLUSTER_TEST,WARNING,0.71,1.58,None
2,LOW_FITNESS,WARNING,0.58,1.0,None
3,LOW_GLB_AMER_SHARPE,WARNING,0.54,1,None
4,LOW_GLB_EMEA_SHARPE,WARNING,0.33,1,None
5,HT_LIQUID_TOPDIV3000_SHARPE,WARNING,1.27,1.7,None
6,HT_INVESTABLE_MAX_TRADE_SHARPE,WARNING,0.97,2.0,None
7,HT_INVESTABLE_MAX_POSITION_SHARPE,WARNING,1.41,2.0,None
8,HT_ORTHOGONAL_RAM_NEUTRALIZATION,WARNING,Subindustry,RAM,None
9,LOW_2Y_SHARPE,WARNING,0.76,1.58,None
